# Stage 1b: Qwen on Kaggle, BOTH T4s (dual-process split) -- runs CONCURRENTLY with local

Splits Stage 1's work (routing, beam-k10 gen, ngram-top10, Qwen's own prefix-cache
teacher-forced scoring, KN5 scoring) across a row-range assigned to THIS notebook
(`ROW_FRAC_START`/`ROW_FRAC_END`, a slice of the full dev/test sets), further split
across both T4s via 2 OS processes (same pattern as `mistral_only_infer.ipynb`).
Runs at the SAME TIME as `scripts/run_qwen_local.py` processing the complementary
slice on the local 4060 -- parallelizes Stage 1 itself instead of only Stage 2.

Output: `dev_qwen_scores_kaggle.jsonl` / `test_qwen_scores_kaggle.jsonl`, same row
SUBSET only (the assigned slice) -- concatenate with local's output (disjoint slice)
before Stage 2.

In [ ]:
import os, glob, json, subprocess, sys, time, selectors

import torch
print("GPUs visible:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i}", torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB")
NUM_SHARDS = 1  # forced -- dual-GPU Qwen crashes on host RAM regardless of LoRA/merge
# (confirmed twice, order-independent -- see status.md). Single-GPU is proven reliable
# and, with the merged model, actually faster solo than the LoRA-loaded version was.

In [ ]:
!pip install -q kenlm
!pip install -q -U torchao

In [ ]:
def find_file(name):
    matches = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert matches, f"{name} not found under /kaggle/input -- check the dataset is attached"
    return matches[0]

def find_by_config(model_type):
    for cfg_path in glob.glob("/kaggle/input/**/config.json", recursive=True):
        try:
            cfg = json.load(open(cfg_path))
        except (json.JSONDecodeError, OSError):
            continue
        if cfg.get("model_type") == model_type:
            return os.path.dirname(cfg_path)
    return None

DEV_PATH = find_file("dev_set_final.csv")
TEST_PATH = find_file("test_set_no_answer_final.csv")
QWEN_MERGED = find_by_config("qwen2")  # pre-merged (LoRA baked in) -- see qwen3b-merged-6h dataset
NGRAM_PATH = find_file("ngram_4_a.bin")
KN5_PATH = find_file("kn5.binary")
KN5_NUM_PATH = find_file("kn5_numbers.binary")
print("DEV_PATH:", DEV_PATH); print("TEST_PATH:", TEST_PATH)
print("QWEN_MERGED:", QWEN_MERGED)
print("NGRAM_PATH:", NGRAM_PATH); print("KN5_PATH:", KN5_PATH); print("KN5_NUM_PATH:", KN5_NUM_PATH)

## Row-range assignment -- Kaggle gets [ROW_FRAC_START, ROW_FRAC_END) of BOTH dev and
test, local gets the complement. Set to cover this notebook's share of the urgent
6h-window run; local's `run_qwen_local.py` must be launched with the matching
complementary `--row-frac-start/--row-frac-end` for the slices to union to the full set
with no gaps/overlap.

In [ ]:
ROW_FRAC_START = 0.207  # local (run_qwen_local.py) takes [0.0, 0.207) concurrently --
# balanced split from real measured rates (local ~1.07 rows/s, kaggle-solo ~4.1 rows/s
# with the merged model): x/1.07 = (1-x)/4.1 -> x~=0.207, both finish ~same time.
ROW_FRAC_END = 1.0
SMALL_BATCH_LIMIT = None  # real run

## Materialize the worker script -- one process/GPU/Qwen-instance, same pattern as
`mistral_only_infer.ipynb`. Inlines `run_qwen_local.py`'s route/gen/ngram/prefix-cache
logic (verbatim) -- no cross-file imports Kaggle can't resolve.

In [ ]:
WORKER_SRC = r'''
import argparse, csv, json, os, re, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, StoppingCriteria
import kenlm

CAT_NUMBER = re.compile(r"[0-9]+")
CAT_WORD = re.compile(r"(?=.*[a-z])[a-z']+", re.IGNORECASE)
def categorize(tok):
    if CAT_NUMBER.fullmatch(tok): return "number"
    if CAT_WORD.fullmatch(tok): return "word"
    return "symbol"
def route(first_letter):
    if first_letter.isalpha(): return "word"
    if first_letter.isdigit(): return "number"
    return "symbol"

def categorize(tok):
    if CAT_NUMBER.fullmatch(tok): return "number"
    if CAT_WORD.fullmatch(tok): return "word"
    return "symbol"

def stratified_subsample(rows, target_n, seed=123):
    import random
    rng = random.Random(seed)
    by_cat = {}
    for i, r in enumerate(rows):
        by_cat.setdefault(categorize(r["answer"]), []).append(i)
    selected = []
    for cat, idx in by_cat.items():
        idx = idx[:]
        rng.shuffle(idx)
        take = round(target_n * len(idx) / len(rows))
        selected.extend(idx[:take])
    rng.shuffle(selected)
    return [rows[i] for i in selected]

def detect_boundary(tokenizer):
    vocab = tokenizer.get_vocab()
    counts = {"\u2581": 0, "\u0120": 0}
    for piece in vocab:
        if piece[:1] in counts: counts[piece[:1]] += 1
    return max(counts, key=counts.get)
def build_letter_masks(tokenizer, boundary, device, vocab_size):
    vocab = tokenizer.get_vocab()
    letters = list("abcdefghijklmnopqrstuvwxyz")
    masks = {c: torch.zeros(vocab_size, dtype=torch.bool) for c in letters}
    for piece, idx in vocab.items():
        if len(piece) > 1 and piece[0] == boundary and piece[1].lower() in masks:
            masks[piece[1].lower()][idx] = True
    return {c: m.to(device) for c, m in masks.items()}
def get_boundary_ids(tokenizer, boundary):
    return [idx for piece, idx in tokenizer.get_vocab().items() if piece.startswith(boundary)]

class FirstTokenLetterMask(LogitsProcessor):
    def __init__(self, letter_mask, prompt_len):
        self.letter_mask = letter_mask; self.prompt_len = prompt_len
    def __call__(self, input_ids, scores):
        if input_ids.shape[1] == self.prompt_len:
            scores = scores.masked_fill(~self.letter_mask, float("-inf"))
        return scores
class StopAtWordBoundary(StoppingCriteria):
    def __init__(self, prompt_len, boundary_ids_tensor, eos_id):
        self.prompt_len = prompt_len; self.boundary_ids_tensor = boundary_ids_tensor; self.eos_id = eos_id
    def __call__(self, input_ids, scores, **kwargs):
        if input_ids.shape[1] <= self.prompt_len + 1:
            return torch.zeros(input_ids.shape[0], dtype=torch.bool, device=input_ids.device)
        last = input_ids[:, -1]
        return torch.isin(last, self.boundary_ids_tensor) | (last == self.eos_id)

@torch.inference_mode()
def predict_topk_batch(model, tokenizer, contexts, letters, masks, boundary_ids_tensor, device, k, max_extra=4):
    enc = tokenizer(contexts, return_tensors="pt", padding=True).to(device)
    letter_mask = torch.stack([masks[l.lower()] for l in letters]).repeat_interleave(k, dim=0)
    prompt_len = enc["input_ids"].shape[1]
    eos_id = tokenizer.eos_token_id
    out = model.generate(**enc, max_new_tokens=max_extra + 1, num_beams=k, num_return_sequences=k,
                          do_sample=False, early_stopping=True,
                          logits_processor=[FirstTokenLetterMask(letter_mask, prompt_len)],
                          stopping_criteria=[StopAtWordBoundary(prompt_len, boundary_ids_tensor, eos_id)],
                          pad_token_id=eos_id)
    generated = out[:, prompt_len:].tolist()
    boundary_set = set(boundary_ids_tensor.tolist())
    results = []
    for b in range(len(contexts)):
        ranked, seen = [], set()
        for beam in range(k):
            row = generated[b * k + beam]
            cut = next((i for i, tid in enumerate(row) if i > 0 and (tid in boundary_set or tid == eos_id)), len(row))
            w = tokenizer.decode(row[:cut]).strip().lower()
            if w and w not in seen:
                seen.add(w); ranked.append(w)
        results.append(ranked)
    return results

def _expand_cache(past_key_values, n):
    if hasattr(past_key_values, "batch_repeat_interleave"):
        past_key_values.batch_repeat_interleave(n); return past_key_values
    if hasattr(past_key_values, "key_cache"):
        for i in range(len(past_key_values.key_cache)):
            past_key_values.key_cache[i] = past_key_values.key_cache[i].repeat_interleave(n, dim=0)
            past_key_values.value_cache[i] = past_key_values.value_cache[i].repeat_interleave(n, dim=0)
        return past_key_values
    return tuple((k.repeat_interleave(n, dim=0), v.repeat_interleave(n, dim=0)) for k, v in past_key_values)

@torch.inference_mode()
def score_candidates_prefix_cache(model, tokenizer, context, candidates, device):
    if not candidates: return []
    ctx_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    cand_id_lists = [tokenizer(" " + cand, add_special_tokens=False)["input_ids"] for cand in candidates]
    prefix_len = len(ctx_ids); n = len(candidates); pad_id = tokenizer.pad_token_id
    ctx_input = torch.tensor([ctx_ids], device=device)
    out1 = model(input_ids=ctx_input, use_cache=True)
    past = _expand_cache(out1.past_key_values, n)
    first_tok_logprob_row = torch.log_softmax(out1.logits[0, -1].float(), dim=-1)
    cand_lens = [len(ids) for ids in cand_id_lists]
    max_len = max(cand_lens)
    cont_ids = torch.full((n, max_len), pad_id, dtype=torch.long)
    cont_mask = torch.zeros((n, max_len), dtype=torch.long)
    for i, ids in enumerate(cand_id_lists):
        cont_ids[i, :len(ids)] = torch.tensor(ids); cont_mask[i, :len(ids)] = 1
    cont_ids, cont_mask = cont_ids.to(device), cont_mask.to(device)
    full_attn_mask = torch.cat([torch.ones(n, prefix_len, dtype=torch.long, device=device), cont_mask], dim=1)
    position_ids = torch.arange(prefix_len, prefix_len + max_len, device=device).unsqueeze(0).expand(n, -1)
    if max_len > 1:
        out2 = model(input_ids=cont_ids, attention_mask=full_attn_mask, past_key_values=past,
                      position_ids=position_ids, use_cache=False)
        cont_logprobs = torch.log_softmax(out2.logits[:, :-1], dim=-1)
    else:
        cont_logprobs = None
    results = []
    for i, ids in enumerate(cand_id_lists):
        lp = first_tok_logprob_row[ids[0]].float().item()
        for t in range(1, len(ids)):
            lp += cont_logprobs[i, t - 1, ids[t]].float().item()
        results.append(lp)
    return results

def load_model_ngram(path):
    import pickle
    with open(path, "rb") as f: m = pickle.load(f)
    return m["n"], m["counts"], m["vocab"], m["id_to_tok"]
def topk_by_letter(context_tokens, letter, n, counts, vocab, id_to_tok, k):
    ids = [vocab.get(t) for t in context_tokens[-(n - 1):]]
    seen, out = set(), []
    for j in range(len(ids), 0, -1):
        ctx_ids = ids[-j:]
        if None in ctx_ids: continue
        d = counts[j].get(tuple(ctx_ids))
        if not d: continue
        for wid, _ in sorted(d.items(), key=lambda kv: -kv[1]):
            w = id_to_tok[wid]
            if w and w[0] == letter and w not in seen:
                seen.add(w); out.append(w)
                if len(out) >= k: return out
    uni = counts[0][()]
    for wid, _ in sorted(uni.items(), key=lambda kv: -kv[1]):
        w = id_to_tok[wid]
        if w and w[0] == letter and w not in seen:
            seen.add(w); out.append(w)
            if len(out) >= k: break
    return out

LN10 = __import__("math").log(10)
class KN5Scorer:
    def __init__(self, path, order=5):
        self.model = kenlm.Model(path); self.order = order
    def score(self, context_tokens, candidate):
        ctx = context_tokens[-(self.order - 1):]
        text = " ".join(ctx + [candidate.lower()])
        log10p, _, _ = list(self.model.full_scores(text, bos=False, eos=False))[-1]
        return log10p * LN10
MAX_NUM_LEN = 8
def pick_length(scorer, context_tokens):
    best_len, best_score = 1, float("-inf")
    for length in range(1, MAX_NUM_LEN + 1):
        s = scorer.score(context_tokens, "1" * length)
        if s > best_score: best_len, best_score = length, s
    return best_len

def load_rows(path, has_answer):
    with open(path, encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    out = []
    for r in rows:
        d = {"context": r["context"], "first_letter": r["first letter"]}
        if has_answer: d["answer"] = r["answer"]
        out.append(d)
    return out

def run_stage1(rows, model, tok, masks, boundary_ids, device, n, counts, vocab, id_to_tok,
                kn_scorer, number_scorer, beam_k, ngram_topk, gen_batch_size, shard, num_shards, name):
    t0 = time.time()
    assigned = [i for i in range(len(rows)) if i % num_shards == shard]
    out_rows = {}
    word_idx = []
    for i in assigned:
        r = rows[i]
        cat = route(r["first_letter"])
        if cat == "symbol":
            out_rows[i] = {**r, "route": "symbol", "pred": r["first_letter"]}
        elif cat == "number":
            pred = "1" * pick_length(number_scorer, r["context"].split())
            out_rows[i] = {**r, "route": "number", "pred": pred}
        else:
            word_idx.append(i)
    print(f"[{name} shard{shard}] {len(word_idx)}/{len(assigned)} assigned rows routed to word pipeline", flush=True)

    beam_cands = {}
    for start in range(0, len(word_idx), gen_batch_size):
        batch_idx = word_idx[start:start + gen_batch_size]
        contexts = [rows[i]["context"] for i in batch_idx]
        letters = [rows[i]["first_letter"] for i in batch_idx]
        preds = predict_topk_batch(model, tok, contexts, letters, masks, boundary_ids, device, beam_k)
        for i, p in zip(batch_idx, preds): beam_cands[i] = p
        done = start + len(batch_idx)
        if done % (gen_batch_size * 10) == 0 or done == len(word_idx):
            el = time.time() - t0
            print(f"[{name} shard{shard} gen {done}/{len(word_idx)}] {el:.1f}s, {done/max(el,1e-9):.2f} rows/s", flush=True)

    t1 = time.time()
    for j, i in enumerate(word_idx):
        r = rows[i]
        ctx_tokens = r["context"].split()
        ngram_cands = topk_by_letter(ctx_tokens, r["first_letter"], n, counts, vocab, id_to_tok, ngram_topk)
        cands = list(dict.fromkeys((beam_cands.get(i) or []) + ngram_cands))
        if not cands:
            out_rows[i] = {**r, "route": "word", "candidates": [], "qwen_scores": [], "kn5_scores": []}
            continue
        q_scores = score_candidates_prefix_cache(model, tok, r["context"], cands, device)
        kn5_scores = [kn_scorer.score(ctx_tokens, w) for w in cands]
        out_rows[i] = {**r, "route": "word", "candidates": cands, "qwen_scores": q_scores, "kn5_scores": kn5_scores}
        if (j + 1) % 200 == 0 or j + 1 == len(word_idx):
            el = time.time() - t1
            print(f"[{name} shard{shard} qwen-tf {j+1}/{len(word_idx)}] {el:.1f}s, {(j+1)/max(el,1e-9):.2f} rows/s", flush=True)

    print(f"[{name} shard{shard}] done: {len(assigned)} assigned rows in {time.time()-t0:.1f}s", flush=True)
    return [{"index": i, **out_rows[i]} for i in assigned]

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--dev-path", required=True)
    ap.add_argument("--test-path", required=True)
    ap.add_argument("--qwen-merged", required=True)
    ap.add_argument("--ngram", required=True)
    ap.add_argument("--kn-model", required=True)
    ap.add_argument("--number-model", required=True)
    ap.add_argument("--device", required=True)
    ap.add_argument("--shard", type=int, required=True)
    ap.add_argument("--num-shards", type=int, required=True)
    ap.add_argument("--dev-subsample", type=int, default=20000)
    ap.add_argument("--row-frac-start", type=float, default=0.0)
    ap.add_argument("--row-frac-end", type=float, default=1.0)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--beam-k", type=int, default=10)
    ap.add_argument("--ngram-topk", type=int, default=10)
    ap.add_argument("--gen-batch-size", type=int, default=16)
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    n, counts, vocab, id_to_tok = load_model_ngram(args.ngram)
    kn_scorer = KN5Scorer(args.kn_model)
    number_scorer = KN5Scorer(args.number_model)

    tok = AutoTokenizer.from_pretrained(args.qwen_merged)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(args.qwen_merged, dtype=torch.float16, attn_implementation="sdpa",
                                                  low_cpu_mem_usage=True).to(args.device)
    model.eval()
    print(f"[shard{args.shard}] Qwen loaded fp16/sdpa on {args.device} in {time.time()-t0:.1f}s", flush=True)

    vocab_size = model.get_output_embeddings().weight.shape[0]
    boundary = detect_boundary(tok)
    masks = build_letter_masks(tok, boundary, args.device, vocab_size)
    boundary_ids = torch.tensor(get_boundary_ids(tok, boundary), device=args.device)

    dev_full = load_rows(args.dev_path, has_answer=True)
    test_full = load_rows(args.test_path, has_answer=False)
    if args.limit:
        dev_full, test_full = dev_full[:args.limit], test_full[:args.limit]
    elif args.dev_subsample:
        full_n = len(dev_full)
        dev_full = stratified_subsample(dev_full, args.dev_subsample)
        print(f"[shard{args.shard}] dev subsampled: {full_n} -> {len(dev_full)} rows", flush=True)
    s, e = int(args.row_frac_start * len(dev_full)), int(args.row_frac_end * len(dev_full))
    dev_rows = [{"index": i, **dev_full[i]} for i in range(s, e)]
    s, e = int(args.row_frac_start * len(test_full)), int(args.row_frac_end * len(test_full))
    test_rows = [{"index": i, **test_full[i]} for i in range(s, e)]
    print(f"[shard{args.shard}] {len(dev_rows)} dev rows, {len(test_rows)} test rows in this notebook's slice", flush=True)

    dev_out = run_stage1(dev_rows, model, tok, masks, boundary_ids, args.device, n, counts, vocab, id_to_tok,
                          kn_scorer, number_scorer, args.beam_k, args.ngram_topk, args.gen_batch_size,
                          args.shard, args.num_shards, "dev")
    test_out = run_stage1(test_rows, model, tok, masks, boundary_ids, args.device, n, counts, vocab, id_to_tok,
                           kn_scorer, number_scorer, args.beam_k, args.ngram_topk, args.gen_batch_size,
                           args.shard, args.num_shards, "test")

    with open(f"{args.out_dir}/dev_qwen_shard{args.shard}.jsonl", "w", encoding="utf-8") as f:
        for r in dev_out: f.write(json.dumps(r) + "\n")
    with open(f"{args.out_dir}/test_qwen_shard{args.shard}.jsonl", "w", encoding="utf-8") as f:
        for r in test_out: f.write(json.dumps(r) + "\n")
    print(f"[shard{args.shard}] wrote {len(dev_out)} dev + {len(test_out)} test rows", flush=True)

if __name__ == "__main__":
    main()
'''
with open("/kaggle/working/qwen_worker.py", "w", encoding="utf-8") as f:
    f.write(WORKER_SRC)
print("wrote /kaggle/working/qwen_worker.py")

In [ ]:
def launch_shard(shard, device):
    cmd = [sys.executable, "/kaggle/working/qwen_worker.py",
           "--dev-path", DEV_PATH, "--test-path", TEST_PATH,
           "--qwen-merged", QWEN_MERGED,
           "--ngram", NGRAM_PATH, "--kn-model", KN5_PATH, "--number-model", KN5_NUM_PATH,
           "--device", device, "--shard", str(shard), "--num-shards", str(NUM_SHARDS),
           "--row-frac-start", str(ROW_FRAC_START), "--row-frac-end", str(ROW_FRAC_END),
           "--out-dir", "/kaggle/working"]
    if SMALL_BATCH_LIMIT:
        cmd += ["--limit", str(SMALL_BATCH_LIMIT)]
    return subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

t0 = time.time()
procs = [launch_shard(s, f"cuda:{s}") for s in range(NUM_SHARDS)]  # no more stagger needed --
# the PeftModel/LoRA-injection step (the identified RAM spike) is gone now that the model
# is pre-merged; single from_pretrained call per shard, same shape as Mistral's proven load
print(f"launched {len(procs)} shard process(es), one per GPU", flush=True)

sel = selectors.DefaultSelector()
for p in procs:
    sel.register(p.stdout, selectors.EVENT_READ, p)
remaining = set(procs)
while remaining:
    for key, _ in sel.select(timeout=1):
        line = key.fileobj.readline()
        if line:
            print(line.rstrip(), flush=True)
    for p in list(remaining):
        if p.poll() is not None:
            remaining.discard(p)

returncodes = [p.wait() for p in procs]
print(f"\nall shards done in {time.time()-t0:.1f}s, returncodes={returncodes}")
assert all(rc == 0 for rc in returncodes), f"a shard process failed: {returncodes}"

## Merge shards -- writes the FULL row-index-tagged output (sparse: only this
notebook's assigned slice has entries) for later concatenation with local's output.

In [ ]:
def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def merge_shards(name):
    rows = []
    for s in range(NUM_SHARDS):
        rows.extend(load_jsonl(f"/kaggle/working/{name}_qwen_shard{s}.jsonl"))
    rows.sort(key=lambda r: r["index"])
    return rows

dev_out = merge_shards("dev")
test_out = merge_shards("test")
with open("/kaggle/working/dev_qwen_scores_kaggle.jsonl", "w", encoding="utf-8") as f:
    for r in dev_out:
        f.write(json.dumps(r) + "\n")
with open("/kaggle/working/test_qwen_scores_kaggle.jsonl", "w", encoding="utf-8") as f:
    for r in test_out:
        f.write(json.dumps(r) + "\n")
print(f"wrote dev_qwen_scores_kaggle.jsonl ({len(dev_out)}), test_qwen_scores_kaggle.jsonl ({len(test_out)}) "
      f"-- indices {ROW_FRAC_START}..{ROW_FRAC_END} of the full sets "
      f"(SMALL_BATCH_LIMIT={SMALL_BATCH_LIMIT} -- NOT the real run until that's None)")